# Распознавание последовательностей символов

Этот ноутбук содержит полный цикл решения задачи:
1. Генерация данных
2. Обучение моделей (KNN)
3. Распознавание и сегментация
4. Оценка средней точности (Accuracy)

In [1]:
# 1. Импорт библиотек
import os
import random
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

In [2]:
# 2. Определение классов (Генератор, Загрузчик, Компаратор)

class SymbolSequenceGenerator:
    def __init__(self, data_path: str, train_size: float = 0.8, spacing: int = 30):
        self.data_path = data_path
        self.train_size = train_size
        self.spacing = spacing

    def create_sequence_image(self, length: int):
        sequence, images = self._generate_list_of_images(length)
        if not images:
            raise ValueError("No images were generated — check dataset path and structure.")

        total_width = sum(img.shape[1] for img in images) + self.spacing * (len(images) - 1)
        max_height = max(img.shape[0] for img in images) + 10

        # White background
        sequence_img = np.ones((max_height, total_width), dtype=np.uint8) * 255

        x_offset = 0
        for img in images:
            h, w = img.shape
            y_offset = (max_height - h) // 2  # Center vertically
            sequence_img[y_offset:y_offset + h, x_offset:x_offset + w] = img
            x_offset += w + self.spacing

        return sequence, cv.bitwise_not(sequence_img)
    
    def _generate_list_of_images(self, length: int) -> list:
        available_symbols = self._get_available_symbols(self.data_path)
        sequence = random.choices(available_symbols, k=length)
        sequence_normalized = [self._normalize_symbol_name(sym) for sym in sequence]
        
        images = []
        for char in sequence:
            img = self._get_random_symbol_image(char)
            if img is not None:
                images.append(img)
        
        return sequence_normalized, images

    def _get_random_symbol_image(self, folder_name: str):
        symbol_dir = os.path.join(self.data_path, folder_name)
        images = sorted(os.listdir(symbol_dir))
        start_idx = int(len(images) * self.train_size)
        subset = images[start_idx:]
        if not subset:
            subset = images

        random_image = random.choice(subset)
        img_path = os.path.join(symbol_dir, random_image)
        img = cv.imread(img_path, cv.IMREAD_GRAYSCALE)
        if img is None: return None
        return img
    
    @staticmethod
    def _normalize_symbol_name(symbol: str) -> str:
        mapping = {'X': 'x', 'times': '*'}
        return mapping.get(symbol, symbol)
    
    @staticmethod
    def _get_available_symbols(data_path: str) -> list:
        subfolders = [name for name in next(os.walk(data_path))[1] if not name.startswith('.')]
        return subfolders

class DataLoader:
    def load_data(self, data_path: str, train_ratio: float = 0.8):
        X, Y = [], []
        symbols = self._get_available_symbols(data_path)
        print(f"Found symbols: {symbols}")

        for symbol in symbols:
            symbol_dir = os.path.join(data_path, symbol)
            images = sorted(os.listdir(symbol_dir))
            label = self._normalize_symbol_name(symbol)
            for img_name in images:
                img_path = os.path.join(symbol_dir, img_name)
                img_vector = self._load_image(img_path)
                if img_vector is None: continue
                X.append(img_vector)
                Y.append(label)

        X = np.array(X)
        Y = np.array(Y)
        X_train, X_test, Y_train, Y_test = train_test_split(
            X, Y, train_size=train_ratio, stratify=Y, random_state=42
        )
        print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
        return X_train, X_test, Y_train, Y_test
    
    @staticmethod
    def _get_available_symbols(data_path: str) -> list:
        return [name for name in next(os.walk(data_path))[1] if not name.startswith('.')]

    @staticmethod
    def _normalize_symbol_name(symbol: str) -> str:
        mapping = {'X': 'x', 'times': '*'}
        return mapping.get(symbol, symbol)

    @staticmethod
    def _load_image(img_path: str):
        img = cv.imread(img_path, cv.IMREAD_GRAYSCALE)
        if img is None: return None
        return img.flatten()

class ModelComparator:
    def __init__(self):
        self.model_results = {}
        
    def add_model_result(self, model_name: str, model: KNeighborsClassifier, accuracy: float):
        self.model_results[model_name] = {'model': model, 'accuracy': accuracy}
        
    def compare_models(self):
        best_name = max(self.model_results.keys(), key=lambda x: self.model_results[x]['accuracy'])
        best_result = self.model_results[best_name]
        return best_name, best_result['model'], best_result['accuracy']

In [3]:
# 3. Инициализация и Тренировка

# Укажите путь к папке с данными
path_to_folder = './data' 

generator = SymbolSequenceGenerator(path_to_folder)
data_loader = DataLoader()

print("--- Loading Data & Training ---")
X_train, X_test, Y_train, Y_test = data_loader.load_data(path_to_folder)

comparator = ModelComparator()
for k in [1, 3, 5, 7]:
    knn = KNeighborsClassifier(n_neighbors=k, n_jobs=-1)
    knn.fit(X_train, Y_train)
    acc = accuracy_score(Y_test, knn.predict(X_test))
    comparator.add_model_result(f"KNN-{k}", knn, acc)
    print(f"KNN-{k} accuracy: {acc:.4f}")

best_name, best_knn_model, best_accuracy = comparator.compare_models()
print(f"Selected Best Model: {best_name}")

--- Loading Data & Training ---
Found symbols: ['(', ')', '+', ',', '-', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'h', 't', 'times', 'w', 'X', 'y']
Train size: 180232, Test size: 45059
KNN-1 accuracy: 0.9882
KNN-3 accuracy: 0.9199
KNN-5 accuracy: 0.7840
KNN-7 accuracy: 0.7371
Selected Best Model: KNN-1


In [4]:
# 4. Функции обработки изображений (Segmentation & Preprocessing)

def pad_to_size(img, target_size=(45, 45)):
    h, w = img.shape
    target_h, target_w = target_size
    padded = np.zeros(target_size, dtype=np.uint8)
    y_start = (target_h - h) // 2
    x_start = (target_w - w) // 2
    padded[y_start:y_start+h, x_start:x_start+w] = img
    return padded

def pad_and_crop(img, target_size=(45, 45), threshold=10):
    h, w = img.shape
    target_h, target_w = target_size
    
    if h == target_h and w == target_w:
        return img
    
    if h > target_h or w > target_w:
        non_zero_rows = np.where(np.max(img, axis=1) > threshold)[0]
        non_zero_cols = np.where(np.max(img, axis=0) > threshold)[0]
        if len(non_zero_rows) > 0 and len(non_zero_cols) > 0:
            y1, y2 = non_zero_rows[0], non_zero_rows[-1] + 1
            x1, x2 = non_zero_cols[0], non_zero_cols[-1] + 1
            img = img[y1:y2, x1:x2]
        h, w = img.shape
        scale = min(target_h / h, target_w / w)
        if scale < 1.0:
            new_h, new_w = int(h * scale), int(w * scale)
            img = cv.resize(img, (new_w, new_h))
    
    return pad_to_size(img, target_size)

def recognize_sequence_from_image(img_gray, model):
    """
    Принимает ч/б изображение последовательности и модель.
    Возвращает список предсказанных символов.
    """
    # 1. Пороговая обработка
    ret, thresh = cv.threshold(img_gray, 127, 255, cv.THRESH_BINARY)
    
    # 2. Поиск контуров
    contours, _ = cv.findContours(thresh, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    
    crops = []
    for cnt in contours:
        x, y, w, h = cv.boundingRect(cnt)
        if w * h > 50: # Фильтр шума
            crops.append([x, y, w, h])
            
    # Сортируем слева направо
    crops.sort(key=lambda x: x[0])
    
    predictions = []
    
    for x, y, w, h in crops:
        img_crop = img_gray[y:y+h, x:x+w]
        
        # Паддинг и центрирование (как в задании)
        base_size = img_crop.shape[0] + 2, img_crop.shape[1] + 30
        base = np.zeros(base_size, dtype=np.uint8)
        base[1:img_crop.shape[0]+1, 15:img_crop.shape[1]+15] = img_crop
        
        # Финальный кроп под размер обучения (45x45)
        final_img = pad_and_crop(base)
        
        # Инверсия (белые буквы на черном для KNN) и Flatten
        img_vector = cv.bitwise_not(final_img).flatten()
        
        pred = model.predict([img_vector])[0]
        predictions.append(pred)
        
    return predictions

In [5]:
# 5. Логика тестирования средней точности

def evaluate_average_accuracy(generator, model, num_sequences=20, seq_length=10):
    print(f"\n--- Starting Evaluation on {num_sequences} sequences ---")
    accuracies = []
    
    for i in range(num_sequences):
        # 1. Генерация
        true_sequence, img_gray = generator.create_sequence_image(seq_length)
        
        # 2. Распознавание
        predicted_sequence = recognize_sequence_from_image(img_gray, model)
        
        # 3. Сравнение
        match_count = 0
        min_len = min(len(true_sequence), len(predicted_sequence))
        
        for j in range(min_len):
            if true_sequence[j] == predicted_sequence[j]:
                match_count += 1
                
        # Точность для данной последовательности
        # Штрафуем за несовпадение длины
        denominator = max(len(true_sequence), len(predicted_sequence))
        if denominator == 0:
            current_acc = 0
        else:
            current_acc = match_count / denominator
            
        accuracies.append(current_acc)
        
        if i < 3: # Показать примеры для первых трех
            print(f"Seq {i+1}: True: {''.join(true_sequence)} | Pred: {''.join(predicted_sequence)} | Acc: {current_acc:.2f}")

    average_acc = sum(accuracies) / len(accuracies)
    return average_acc

In [7]:
# 6. Запуск итогового теста

num_test_sequences = 10  # Количество последовательностей для теста
seq_len = 10             # Длина одной последовательности

# Убедитесь, что best_knn_model обучена (запущен шаг 3)
if 'best_knn_model' in locals():
    avg_accuracy = evaluate_average_accuracy(generator, best_knn_model, num_test_sequences, seq_len)

    print("-" * 30)
    print(f"Average Accuracy over {num_test_sequences} sequences: {avg_accuracy:.4f}")
    print("-" * 30)
else:
    print("Сначала выполните ячейку с обучением моделей!")


--- Starting Evaluation on 10 sequences ---
Seq 1: True: 4401(*,3w* | Pred: 4401(*,3w* | Acc: 1.00
Seq 2: True: 3y-42278h) | Pred: 3y-42278h) | Acc: 1.00
Seq 3: True: 78*0)(*835 | Pred: 78*0)(*83(- | Acc: 0.82
------------------------------
Average Accuracy over 10 sequences: 0.6245
------------------------------
